In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------------
# SETTINGS
# --------------------------------------------------------
base_dir = "data/2-output-data"

scenarios = ["s2", "s3"]
scenario_labels = {"s2": "S2", "s3": "S3"}
scenario_colors = {"S2": "#B47CC7", "S3": "#6ABF69"}

pollutant_labels = {
    "ug_PM25_RH50": "PM2.5",
    "ug_NO2": "NO2"
}

years = ["2030", "2050"]

param_titles = {
    "MortalityAvoided": "Total Mortality Avoided (>30y)",
    "YLG": "Years of Life Gained (>30y)",
    "LE_months": "Life expectancy gained (months, >30y)"
}

# --------------------------------------------------------
# BUILD DATAFRAME FROM SAVED FILES
# --------------------------------------------------------
records = []

for scenario in scenarios:
    for year in years:
        for pollutant in pollutant_labels.keys():

            filepath = os.path.join(
                base_dir, scenario, pollutant, year, "mortality_chimere_mc_sensitivity.csv"
            )

            if not os.path.exists(filepath):
                print(f"⚠ Missing file: {filepath}")
                continue

            df = pd.read_csv(filepath)

            # --- Mortality avoided ---
            mort_med = df["mortpol_age"].sum()
            mort_lci = df["mortpol_LCI"].sum()
            mort_uci = df["mortpol_UCI"].sum()

            records.append([
                year, scenario_labels[scenario], pollutant_labels[pollutant],
                "MortalityAvoided", mort_med, mort_lci, mort_uci
            ])

            # --- YLG (life-table based, stored once per file) ---
            ylg_med = df["YLG_total"].iloc[0]
            ylg_lci = df["YLG_total_LCI"].iloc[0]
            ylg_uci = df["YLG_total_UCI"].iloc[0]

            records.append([
                year, scenario_labels[scenario], pollutant_labels[pollutant],
                "YLG", ylg_med, ylg_lci, ylg_uci
            ])

            # --- Life expectancy gain (months) ---
            le_med = df["overall_LifeTable_LEgain_mo"].iloc[0]
            le_lci = df["overall_LifeTable_LEgain_mo_LCI"].iloc[0]
            le_uci = df["overall_LifeTable_LEgain_mo_UCI"].iloc[0]

            records.append([
                year, scenario_labels[scenario], pollutant_labels[pollutant],
                "LE_months", le_med, le_lci, le_uci
            ])

df_plot = pd.DataFrame(
    records,
    columns=["Year", "Scenario", "Pollutant", "Parameter", "Median", "Min", "Max"]
)

In [ ]:
# --------------------------------------------------------
# PLOTTING: Bar plots with dotted LCI / UCI
# --------------------------------------------------------
width = 0.20
x = np.arange(len(years))

output_dir = base_dir
os.makedirs(output_dir, exist_ok=True)

for pollutant in ["PM2.5", "NO2"]:
    for param in ["MortalityAvoided", "YLG", "LE_months"]:

        df_sub = df_plot[
            (df_plot["Pollutant"] == pollutant) &
            (df_plot["Parameter"] == param)
            ]

        fig, ax = plt.subplots(figsize=(7, 6), dpi=140)

        for i, scenario in enumerate(["S2", "S3"]):
            sub = df_sub[df_sub["Scenario"] == scenario]

            positions = x + (i - 0.55) * width

            medians = [sub[sub["Year"] == yr]["Median"].values[0] for yr in years]
            lcis = [sub[sub["Year"] == yr]["Min"].values[0] for yr in years]
            ucis = [sub[sub["Year"] == yr]["Max"].values[0] for yr in years]

            # --- Median bars ---
            ax.bar(
                positions,
                medians,
                width,
                color=scenario_colors[scenario],
                edgecolor="black",
                linewidth=0.9,
                alpha=0.95,
                label=scenario
            )

            # --- Dotted LCI–UCI vertical lines ---
            for pos, lo, hi in zip(positions, lcis, ucis):
                ax.vlines(
                    pos,
                    lo,
                    hi,
                    linestyles="dotted",
                    linewidth=1.5,
                    color="black",
                    alpha=0.95
                )

        ax.set_xticks(x)
        ax.set_xticklabels(years, fontsize=13)
        ax.set_ylabel(param_titles[param], fontsize=13)
        ax.set_title(f"{param_titles[param]} — {pollutant}", fontsize=15, weight="bold")
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)
        ax.legend(title="Scenario")

        plt.tight_layout()
        outpath = os.path.join(output_dir, f"sensitivity_{param}_{pollutant}_S2_S3.png")
        plt.savefig(outpath, dpi=300, bbox_inches="tight")
        plt.close()

print("Done — Bar plots generated automatically from saved HIA outputs")


In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------------------------------
# SETTINGS
# --------------------------------------------------------
base_dir = "data/2-output-data"

scenarios = ["s2", "s3"]
scenario_labels = {"s2": "S2", "s3": "S3"}
scenario_colors = {"S2": "#B47CC7", "S3": "#6ABF69"}

pollutant_labels = {
    "ug_PM25_RH50": "PM2.5",
    "ug_NO2": "NO2"
}

years = ["2030", "2050"]

param_titles = {
    "avoided_cases_central": "Total Avoided Cases",
    "YLD_central": "Years Lived with Disability (YLD)",
    "YLL_central": "Years of Life Lost (YLL)",
    "DALY_central": "Disability-Adjusted Life Years (DALY)",
}

# --------------------------------------------------------
# BUILD DATAFRAME FROM SAVED FILES
# --------------------------------------------------------
records = []

for scenario in scenarios:
    for year in years:
        for pollutant in pollutant_labels.keys():
            filepath = os.path.join(
                base_dir, scenario, pollutant, year, "morbidity_results.csv"
            )

            if not os.path.exists(filepath):
                print(f"⚠ Missing file: {filepath}")
                continue

            df = pd.read_csv(filepath)

            # Group by disease to maintain disease-level granularity
            disease_groups = df.groupby("disease")

            for disease_name, group in disease_groups:
                # --- Morbidity avoided ---
                mort_med = group["avoided_cases_central"].sum()
                mort_lci = group["avoided_cases_low"].sum()
                mort_uci = group["avoided_cases_high"].sum()

                records.append([
                    year, scenario_labels[scenario], pollutant_labels[pollutant],
                    disease_name, "CasesAvoided", mort_med, mort_lci, mort_uci
                ])

                # --- YLD ---
                yld_med = group["YLD_central"].sum()
                yld_lci = group["YLD_low"].sum()
                yld_uci = group["YLD_high"].sum()

                records.append([
                    year, scenario_labels[scenario], pollutant_labels[pollutant],
                    disease_name, "YLD", yld_med, yld_lci, yld_uci
                ])

                # --- DALYs averted ---
                daly_med = group["DALY_central"].sum()
                daly_lci = group["DALY_low"].sum()
                daly_uci = group["DALY_high"].sum()
                records.append([
                    year, scenario_labels[scenario], pollutant_labels[pollutant],
                    disease_name, "DALY_Averted", daly_med, daly_lci, daly_uci
                ])

                # --YLL (for DALY) ---
                yll_med = group["YLL_central"].sum()
                yll_lci = group["YLL_low"].sum()
                yll_uci = group["YLL_high"].sum()
                records.append([
                    year, scenario_labels[scenario], pollutant_labels[pollutant],
                    disease_name, "YLL", yll_med, yll_lci, yll_uci
                ])

df_plot_morb = pd.DataFrame(
    records,
    columns=["Year", "Scenario", "Pollutant", "Disease", "Parameter", "Median", "Min", "Max"]
)

In [ ]:
# --------------------------------------------------------
# PLOTTING: Bar plots with dotted LCI / UCI for Morbidity
# --------------------------------------------------------
width = 0.35
output_dir = base_dir
os.makedirs(output_dir, exist_ok=True)

morb_titles = {
    "CasesAvoided": "Avoided Cases",
    "YLD": "Years Lived with Disability (YLD)"
}

for pollutant in ["PM2.5", "NO2"]:
    for param in ["CasesAvoided", "YLD"]:

        df_sub = df_plot_morb[
            (df_plot_morb["Pollutant"] == pollutant) &
            (df_plot_morb["Parameter"] == param)
            ].copy()

        if df_sub.empty:
            continue

        # Sort by Year then Disease to ensure 2030 comes first
        df_sub = df_sub.sort_values(["Year", "Disease"])

        # Unique categories for x-axis
        diseases = df_sub["Disease"].unique()
        categories = []
        for yr in years:
            for dis in diseases:
                categories.append(f"{yr}\n{dis}")

        x = np.arange(len(categories))
        fig, ax = plt.subplots(figsize=(14, 7), dpi=140)

        for i, scenario in enumerate(["S2", "S3"]):
            # Filter for scenario and align with categories
            medians, lcis, ucis = [], [], []

            for cat in categories:
                yr_val, dis_val = cat.split("\n")
                match = df_sub[
                    (df_sub["Scenario"] == scenario) &
                    (df_sub["Year"] == yr_val) &
                    (df_sub["Disease"] == dis_val)
                    ]

                if not match.empty:
                    medians.append(match["Median"].values[0])
                    lcis.append(match["Min"].values[0])
                    ucis.append(match["Max"].values[0])
                else:
                    medians.append(0)
                    lcis.append(0)
                    ucis.append(0)

            positions = x + (i - 0.5) * width

            # --- Median bars ---
            ax.bar(
                positions,
                medians,
                width,
                color=scenario_colors[scenario],
                edgecolor="black",
                linewidth=0.9,
                alpha=0.95,
                label=scenario
            )

            # --- Dotted LCI–UCI vertical lines ---
            for pos, lo, hi in zip(positions, lcis, ucis):
                ax.vlines(
                    pos,
                    lo,
                    hi,
                    linestyles="dotted",
                    linewidth=1.5,
                    color="black",
                    alpha=0.95
                )

        # Add vertical line to separate 2030 and 2050
        sep_pos = len(diseases) - 0.5
        ax.axvline(x=sep_pos, color='grey', linestyle='--', linewidth=1.5, alpha=0.7)

        # Set Primary X-axis (Diseases)
        ax.set_xticks(x)
        ax.set_xticklabels([cat.split("\n")[1] for cat in categories], fontsize=10, rotation=45, ha="right")

        # Add Secondary X-axis for Years
        for i, yr in enumerate(years):
            center_pos = (len(diseases) - 1) / 2 + i * len(diseases)
            ax.text(center_pos, -0.15, yr, transform=ax.get_xaxis_transform(),
                    ha='center', va='top', fontsize=12, weight='bold')

        ax.set_ylabel(morb_titles.get(param, param), fontsize=13)
        ax.set_title(f"{morb_titles.get(param, param)} by Disease and Year — {pollutant}", fontsize=15, weight="bold")
        ax.grid(axis="y", linestyle="--", alpha=0.5)
        ax.set_axisbelow(True)
        ax.legend(title="Scenario")

        plt.tight_layout()
        outpath = os.path.join(output_dir, f"morbidity_{param}_{pollutant}_S2_S3.png")
        plt.savefig(outpath, dpi=300, bbox_inches="tight")
        plt.show()

print("Done — Morbidity bar plots generated with separated years")


In [ ]:
# Plot economic summary (bars + component shading for intangible)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd
import numpy as np

path_fichier_expo = os.path.join("data", "2-output-data")
out_file = os.path.join(path_fichier_expo, "economic_summary_morbidity_all.csv")

if os.path.exists(out_file):
    # Load the economic results
    total_summary_df = pd.read_csv(out_file)

    # Normalize key columns
    total_summary_df["Scenario"] = total_summary_df["Scenario"].astype(str).str.upper().str.strip()
    total_summary_df["Disease"] = total_summary_df["Disease"].astype(str).str.upper().str.strip()
    if "Pollutant" in total_summary_df.columns:
        total_summary_df["Pollutant"] = total_summary_df["Pollutant"].astype(str).str.strip()

    # Define colors for scenarios
    scenario_colors = {"S2": "#B47CC7", "S3": "#6ABF69"}

    # Ensure correct datatypes
    total_summary_df["Year"] = pd.to_numeric(total_summary_df["Year"], errors="coerce").astype("Int64")

    # Direct medical costs
    for c in ["Direct Med Cost (M€)", "Direct Med Cost LCI (M€)", "Direct Med Cost UCI (M€)"]:
        if c in total_summary_df.columns:
            total_summary_df[c] = pd.to_numeric(total_summary_df[c], errors="coerce")
        else:
            total_summary_df[c] = 0.0

    # Intangible components (preferred) OR total (fallback)
    has_yld = "Intangible Cost YLD (M€)" in total_summary_df.columns
    has_yll = "Intangible Cost YLL (M€)" in total_summary_df.columns
    has_total_be = "Intangible Cost Total (B€)" in total_summary_df.columns

    for c in [
        "Intangible Cost YLD (M€)",
        "Intangible Cost YLD LCI (M€)",
        "Intangible Cost YLD UCI (M€)",
        "Intangible Cost YLL (M€)",
        "Intangible Cost YLL LCI (M€)",
        "Intangible Cost YLL UCI (M€)",
    ]:
        if c in total_summary_df.columns:
            total_summary_df[c] = pd.to_numeric(total_summary_df[c], errors="coerce")
        else:
            total_summary_df[c] = 0.0

    if has_total_be:
        total_summary_df["Intangible Cost Total (B€)"] = pd.to_numeric(
            total_summary_df["Intangible Cost Total (B€)"], errors="coerce"
        )
        total_summary_df["Intangible Cost Total LCI (B€)"] = pd.to_numeric(
            total_summary_df.get("Intangible Cost Total LCI (B€)"), errors="coerce"
        )
        total_summary_df["Intangible Cost Total UCI (B€)"] = pd.to_numeric(
            total_summary_df.get("Intangible Cost Total UCI (B€)"), errors="coerce"
        )
    else:
        total_summary_df["Intangible Cost Total (B€)"] = 0.0
        total_summary_df["Intangible Cost Total LCI (B€)"] = 0.0
        total_summary_df["Intangible Cost Total UCI (B€)"] = 0.0

    # Filter for TOTAL rows and aggregate by Year and Scenario
    plot_df = (
        total_summary_df[total_summary_df["Disease"] == "TOTAL"]
        .groupby(["Year", "Scenario"], dropna=False)
        .agg({
            "Direct Med Cost (M€)": "sum",
            "Direct Med Cost LCI (M€)": "sum",
            "Direct Med Cost UCI (M€)": "sum",
            "Intangible Cost YLD (M€)": "sum",
            "Intangible Cost YLD LCI (M€)": "sum",
            "Intangible Cost YLD UCI (M€)": "sum",
            "Intangible Cost YLL (M€)": "sum",
            "Intangible Cost YLL LCI (M€)": "sum",
            "Intangible Cost YLL UCI (M€)": "sum",
            "Intangible Cost Total (B€)": "sum",
            "Intangible Cost Total LCI (B€)": "sum",
            "Intangible Cost Total UCI (B€)": "sum",
        })
        .reset_index()
    )

    # Keep only expected scenarios (and order) + order years
    scenario_order = ["S2", "S3"]
    plot_df = plot_df[plot_df["Scenario"].isin(scenario_order)].copy()
    plot_df = plot_df.sort_values(
        ["Year", "Scenario"],
        key=lambda s: s if s.name != "Scenario" else s.map({k: i for i, k in enumerate(scenario_order)}),
    )

    # Create a complete grid of Year x Scenario so both years always plot (even if one combo missing)
    years_order = sorted([int(y) for y in plot_df["Year"].dropna().unique().tolist()])
    full_index = pd.MultiIndex.from_product([years_order, scenario_order], names=["Year", "Scenario"])
    plot_df = (
        plot_df.set_index(["Year", "Scenario"])
        .reindex(full_index)
        .fillna(0)
        .reset_index()
    )

    # Label for x-axis
    plot_df["Label"] = plot_df["Year"].astype(int).astype(str) + " " + plot_df["Scenario"]
    labels = plot_df["Label"].tolist()
    x_pos = np.arange(len(labels))

    # Intangible: compute components in B€ (preferred: YLD+YLL from M€; fallback: use total B€ only)
    plot_df["Intangible YLD (B€)"] = plot_df["Intangible Cost YLD (M€)"] / 1000.0
    plot_df["Intangible YLL (B€)"] = plot_df["Intangible Cost YLL (M€)"] / 1000.0
    plot_df["Intangible Total Components (B€)"] = plot_df["Intangible YLD (B€)"] + plot_df["Intangible YLL (B€)"]
    plot_df["Use Components"] = (plot_df["Intangible Total Components (B€)"] > 0).astype(int)

    # If components are not available, fallback to total B€
    plot_df["Intangible Total (B€)"] = np.where(
        plot_df["Use Components"] == 1,
        plot_df["Intangible Total Components (B€)"],
        plot_df["Intangible Cost Total (B€)"],
    )

    # Build LCI/UCI for intangible totals in B€
    plot_df["Intangible Total LCI (B€)"] = np.where(
        plot_df["Use Components"] == 1,
        (plot_df["Intangible Cost YLD LCI (M€)"] + plot_df["Intangible Cost YLL LCI (M€)"]) / 1000.0,
        plot_df["Intangible Cost Total LCI (B€)"],
    )
    plot_df["Intangible Total UCI (B€)"] = np.where(
        plot_df["Use Components"] == 1,
        (plot_df["Intangible Cost YLD UCI (M€)"] + plot_df["Intangible Cost YLL UCI (M€)"]) / 1000.0,
        plot_df["Intangible Cost Total UCI (B€)"],
    )

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 14), sharex=True)

    max_med_cost = float(plot_df["Direct Med Cost (M€)"].max()) if len(plot_df) else 0.0
    max_int_cost = float(plot_df["Intangible Total (B€)"].max()) if len(plot_df) else 0.0

    # --- Direct Medical Cost (vertical bars) ---
    for i in range(len(plot_df)):
        row = plot_df.iloc[i]
        color = scenario_colors.get(row["Scenario"], "gray")
        val = float(row["Direct Med Cost (M€)"])
        lci = float(row["Direct Med Cost LCI (M€)"])
        uci = float(row["Direct Med Cost UCI (M€)"])

        ax1.bar(i, val, color=color, alpha=0.85, width=0.7, edgecolor="black", linewidth=0.4)
        ax1.text(
            i,
            val + (max_med_cost * 0.005 if max_med_cost else 0.02),
            f"{val:.0f} ({lci:.0f} - {uci:.0f}) M€",
            ha="center",
            va="bottom",
            rotation=0,
            fontweight="bold",
            fontsize=12,
        )

    ax1.set_title("Direct Medical Costs (M€)", fontsize=18, fontweight="bold")
    ax1.set_ylabel("Value (M€)", fontsize=14)
    ax1.grid(axis="y", linestyle="--", alpha=0.6)

    # --- Intangible Costs (hatching for components) ---
    hatch_yld = "///"  # morbidity
    hatch_yll = "xxx"  # mortality

    for i in range(len(plot_df)):
        row = plot_df.iloc[i]
        color = scenario_colors.get(row["Scenario"], "gray")

        # If components exist, use stacked + hatches; else show total as solid
        if int(row["Use Components"]) == 1:
            yld = float(row["Intangible YLD (B€)"])
            yll = float(row["Intangible YLL (B€)"])
            total = yld + yll

            ax2.bar(i, yld, color=color, alpha=0.55, width=0.7, edgecolor="black", linewidth=0.4, hatch=hatch_yld)
            ax2.bar(
                i,
                yll,
                bottom=yld,
                color=color,
                alpha=0.85,
                width=0.7,
                edgecolor="black",
                linewidth=0.4,
                hatch=hatch_yll,
            )
        else:
            total = float(row["Intangible Total (B€)"])
            ax2.bar(i, total, color=color, alpha=0.85, width=0.7, edgecolor="black", linewidth=0.4)

        lci_int = float(row["Intangible Total LCI (B€)"])
        uci_int = float(row["Intangible Total UCI (B€)"])

        ax2.text(
            i,
            total + (max_int_cost * 0.005 if max_int_cost else 0.002),
            f"{total:.0f} ({lci_int:.0f} - {uci_int:.0f}) B€",
            ha="center",
            va="bottom",
            rotation=0,
            fontweight="bold",
            fontsize=12,
        )

    ax2.set_title("Intangible Costs (B€): Morbidity (YLD) + Mortality (YLL)", fontsize=18, fontweight="bold")
    ax2.set_ylabel("Value (B€)", fontsize=14)
    ax2.grid(axis="y", linestyle="--", alpha=0.6)

    # X axis labels
    ax2.set_xticks(x_pos)
    ax2.set_xticklabels(labels, fontsize=14, fontweight="bold", rotation=0)

    # Legend
    from matplotlib.lines import Line2D
    from matplotlib.patches import Patch

    legend_scen = [Line2D([0], [0], color=c, lw=8, label=s) for s, c in scenario_colors.items()]
    legend_comp = [
        Patch(facecolor="white", edgecolor="black", hatch=hatch_yld, label="Intangible Cost morbidity (YLD)"),
        Patch(facecolor="white", edgecolor="black", hatch=hatch_yll, label="Intangible Cost mortality (YLL)"),
    ]

    fig.legend(
        handles=legend_scen + legend_comp,
        loc="upper center",
        title="Scenario & Components",
        bbox_to_anchor=(0.5, 0.98),
        ncol=6,
        fontsize=12,
        title_fontsize=13,
    )

    plt.suptitle("Economic Summary: Direct Medical vs Intangible Costs", fontsize=20, fontweight="bold", y=1.02)
    plt.tight_layout(rect=[0, 0.03, 1, 0.95])

    # Save the plot
    save_path = os.path.join(path_fichier_expo, "economic_summary_plot.png")
    plt.savefig(save_path, dpi=300, bbox_inches="tight")

    plt.show()
else:
    print(f"File not found: {out_file}")

In [ ]:
# economic cost plot by diseases (separate plots for each pollutant)
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd
import numpy as np

path_fichier_expo = os.path.join("data", "2-output-data")
out_file = os.path.join(path_fichier_expo, "economic_summary_morbidity_all.csv")

if os.path.exists(out_file):
    total_summary_df = pd.read_csv(out_file)

    # Normalize key columns
    total_summary_df["Scenario"] = total_summary_df["Scenario"].astype(str).str.upper().str.strip()
    total_summary_df["Disease"] = total_summary_df["Disease"].astype(str).str.upper().str.strip()
    if "Pollutant" in total_summary_df.columns:
        total_summary_df["Pollutant"] = total_summary_df["Pollutant"].astype(str).str.strip()

    # Define colors for scenarios
    scenario_colors = {"S2": "#B47CC7", "S3": "#6ABF69"}
    scenario_order = ["S2", "S3"]

    # Ensure correct datatypes (create missing cols as 0)
    total_summary_df["Year"] = pd.to_numeric(total_summary_df["Year"], errors="coerce").astype("Int64")

    num_cols = [
        "Direct Med Cost (M€)", "Direct Med Cost LCI (M€)", "Direct Med Cost UCI (M€)",
        "Intangible Cost YLD (M€)", "Intangible Cost YLD LCI (M€)", "Intangible Cost YLD UCI (M€)",
        "Intangible Cost YLL (M€)", "Intangible Cost YLL LCI (M€)", "Intangible Cost YLL UCI (M€)",
        "Intangible Cost Total (B€)", "Intangible Cost Total LCI (B€)", "Intangible Cost Total UCI (B€)",
    ]
    for c in num_cols:
        if c in total_summary_df.columns:
            total_summary_df[c] = pd.to_numeric(total_summary_df[c], errors="coerce").fillna(0.0)
        else:
            total_summary_df[c] = 0.0

    # Decide pollutants to plot (prefer 2 pollutants if present)
    if "Pollutant" in total_summary_df.columns:
        available = total_summary_df["Pollutant"].dropna().unique().tolist()
        preferred = [p for p in ["PM2.5", "NO2"] if p in available]
        pollutants = preferred if preferred else available
    else:
        pollutants = ["All Pollutants"]

    # Generate separate plots per pollutant (saved to disk; no multiple displays)
    for pollutant in pollutants:
        if pollutant == "All Pollutants":
            pollutant_df = total_summary_df.copy()
            pollutant_safe = "ALL"
        else:
            pollutant_df = total_summary_df[total_summary_df["Pollutant"] == pollutant].copy()
            pollutant_safe = str(pollutant).replace("/", "_").replace("\\", "_").replace(" ", "_")

        if pollutant_df.empty:
            print(f"No data available for {pollutant}. Skipping plot.")
            continue

        plot_pollut_df = (
            pollutant_df[pollutant_df["Disease"] == "TOTAL"]
            .groupby(["Year", "Scenario"], dropna=False)
            .agg({
                "Direct Med Cost (M€)": "sum",
                "Direct Med Cost LCI (M€)": "sum",
                "Direct Med Cost UCI (M€)": "sum",
                "Intangible Cost YLD (M€)": "sum",
                "Intangible Cost YLD LCI (M€)": "sum",
                "Intangible Cost YLD UCI (M€)": "sum",
                "Intangible Cost YLL (M€)": "sum",
                "Intangible Cost YLL LCI (M€)": "sum",
                "Intangible Cost YLL UCI (M€)": "sum",
                "Intangible Cost Total (B€)": "sum",
                "Intangible Cost Total LCI (B€)": "sum",
                "Intangible Cost Total UCI (B€)": "sum",
            })
            .reset_index()
        )

        if plot_pollut_df.empty:
            print(f"No TOTAL rows available for {pollutant}. Skipping plot.")
            continue

        plot_pollut_df = plot_pollut_df[plot_pollut_df["Scenario"].isin(scenario_order)].copy()

        years_order = sorted([int(y) for y in plot_pollut_df["Year"].dropna().unique().tolist()])
        if not years_order:
            print(f"No valid years available for {pollutant}. Skipping plot.")
            continue

        plot_pollut_df = plot_pollut_df.sort_values(
            ["Year", "Scenario"],
            key=lambda s: s if s.name != "Scenario" else s.map({k: i for i, k in enumerate(scenario_order)}),
        )

        full_index = pd.MultiIndex.from_product([years_order, scenario_order], names=["Year", "Scenario"])
        plot_pollut_df = (
            plot_pollut_df.set_index(["Year", "Scenario"])
            .reindex(full_index)
            .fillna(0)
            .reset_index()
        )

        plot_pollut_df["Label"] = plot_pollut_df["Year"].astype(int).astype(str) + " " + plot_pollut_df["Scenario"]
        labels = plot_pollut_df["Label"].tolist()
        x_pos = np.arange(len(labels))

        # Intangible costs in B€: prefer components (YLD+YLL) if present, else fallback total B€
        plot_pollut_df["Intangible YLD (B€)"] = plot_pollut_df["Intangible Cost YLD (M€)"] / 1000.0
        plot_pollut_df["Intangible YLL (B€)"] = plot_pollut_df["Intangible Cost YLL (M€)"] / 1000.0
        plot_pollut_df["Intangible Total Components (B€)"] = (
                plot_pollut_df["Intangible YLD (B€)"] + plot_pollut_df["Intangible YLL (B€)"]
        )
        plot_pollut_df["Use Components"] = (plot_pollut_df["Intangible Total Components (B€)"] > 0).astype(int)

        plot_pollut_df["Intangible Total (B€)"] = np.where(
            plot_pollut_df["Use Components"] == 1,
            plot_pollut_df["Intangible Total Components (B€)"],
            plot_pollut_df["Intangible Cost Total (B€)"],
        )
        plot_pollut_df["Intangible Total LCI (B€)"] = np.where(
            plot_pollut_df["Use Components"] == 1,
            (plot_pollut_df["Intangible Cost YLD LCI (M€)"] + plot_pollut_df["Intangible Cost YLL LCI (M€)"]) / 1000.0,
            plot_pollut_df["Intangible Cost Total LCI (B€)"],
        )
        plot_pollut_df["Intangible Total UCI (B€)"] = np.where(
            plot_pollut_df["Use Components"] == 1,
            (plot_pollut_df["Intangible Cost YLD UCI (M€)"] + plot_pollut_df["Intangible Cost YLL UCI (M€)"]) / 1000.0,
            plot_pollut_df["Intangible Cost Total UCI (B€)"],
        )

        # --- Generate plots ---
        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(18, 14), sharex=True)
        max_med_cost = float(plot_pollut_df["Direct Med Cost (M€)"].max()) if len(plot_pollut_df) else 0.0
        max_int_cost = float(plot_pollut_df["Intangible Total (B€)"].max()) if len(plot_pollut_df) else 0.0

        # Bars for Direct Medical Costs
        for i in range(len(plot_pollut_df)):
            row = plot_pollut_df.iloc[i]
            color = scenario_colors.get(row["Scenario"], "gray")
            val = float(row["Direct Med Cost (M€)"])
            lci = float(row["Direct Med Cost LCI (M€)"])
            uci = float(row["Direct Med Cost UCI (M€)"])
            ax1.bar(i, val, color=color, alpha=0.85, width=0.7, edgecolor="black", linewidth=0.4)
            ax1.text(
                i,
                val + (max_med_cost * 0.005 if max_med_cost else 0.02),
                f"{val:.0f} ({lci:.0f} - {uci:.0f}) M€",
                ha="center",
                va="bottom",
                rotation=0,
                fontweight="bold",
                fontsize=12,
            )
        ax1.set_title(f"Direct Medical Costs (M€) — {pollutant}", fontsize=18, fontweight="bold")
        ax1.set_ylabel("Value (M€)", fontsize=14)
        ax1.grid(axis="y", linestyle="--", alpha=0.6)

        # Bars for Intangible Costs
        hatch_yld = "///"
        hatch_yll = "xxx"
        for i in range(len(plot_pollut_df)):
            row = plot_pollut_df.iloc[i]
            color = scenario_colors.get(row["Scenario"], "gray")
            if int(row["Use Components"]) == 1:
                yld = float(row["Intangible YLD (B€)"])
                yll = float(row["Intangible YLL (B€)"])
                total = yld + yll
                ax2.bar(i, yld, color=color, alpha=0.55, width=0.7, edgecolor="black", linewidth=0.4, hatch=hatch_yld)
                ax2.bar(
                    i,
                    yll,
                    bottom=yld,
                    color=color,
                    alpha=0.85,
                    width=0.7,
                    edgecolor="black",
                    linewidth=0.4,
                    hatch=hatch_yll,
                )
            else:
                total = float(row["Intangible Total (B€)"])
                ax2.bar(i, total, color=color, alpha=0.85, width=0.7, edgecolor="black", linewidth=0.4)

            lci_int = float(row["Intangible Total LCI (B€)"])
            uci_int = float(row["Intangible Total UCI (B€)"])
            ax2.text(
                i,
                total + (max_int_cost * 0.005 if max_int_cost else 0.002),
                f"{total:.0f} ({lci_int:.0f} - {uci_int:.0f}) B€",
                ha="center",
                va="bottom",
                rotation=0,
                fontweight="bold",
                fontsize=12,
            )

        ax2.set_title(f"Intangible Costs (B€): Morbidity + Mortality — {pollutant}", fontsize=18, fontweight="bold")
        ax2.set_ylabel("Value (B€)", fontsize=14)
        ax2.grid(axis="y", linestyle="--", alpha=0.6)
        ax2.set_xticks(x_pos)
        ax2.set_xticklabels(labels, fontsize=14, fontweight="bold", rotation=0)

        # Legend
        from matplotlib.lines import Line2D
        from matplotlib.patches import Patch

        legend_scen = [Line2D([0], [0], color=c, lw=8, label=s) for s, c in scenario_colors.items()]
        legend_comp = [
            Patch(facecolor="white", edgecolor="black", hatch=hatch_yld, label="Intangible Cost morbidity (YLD)"),
            Patch(facecolor="white", edgecolor="black", hatch=hatch_yll, label="Intangible Cost mortality (YLL)"),
        ]

        fig.legend(
            handles=legend_scen + legend_comp,
            loc="upper center",
            title="Scenario & Components",
            bbox_to_anchor=(0.5, 0.98),
            ncol=6,
            fontsize=12,
            title_fontsize=13,
        )

        fig.suptitle(f"Economic Summary — {pollutant}", fontsize=20, fontweight="bold", y=1.02)
        plt.tight_layout(rect=[0, 0.03, 1, 0.95])

        save_path = os.path.join(path_fichier_expo, f"economic_summary_plot_{pollutant_safe}.png")
        plt.savefig(save_path, dpi=300, bbox_inches="tight")
        plt.close(fig)

        print(f"Plot saved for pollutant {pollutant}: {save_path}")
else:
    print(f"File not found: {out_file}")


In [ ]:
#economic cost plot by diseases
import matplotlib.pyplot as plt
import seaborn as sns
import os
import pandas as pd
import numpy as np

path_fichier_expo = os.path.join("data", "2-output-data")
out_file = os.path.join(path_fichier_expo, "economic_summary_morbidity_all.csv")

if os.path.exists(out_file):
    # Load the economic results
    total_summary_df = pd.read_csv(out_file)

    # Normalize Scenario column
    total_summary_df["Scenario"] = total_summary_df["Scenario"].str.upper()

    # Define colors for scenarios
    scenario_colors = {"S2": "#B47CC7", "S3": "#6ABF69"}

    # Ensure correct datatypes
    total_summary_df["Year"] = pd.to_numeric(total_summary_df["Year"], errors='coerce')
    total_summary_df["Direct Med Cost (M€)"] = pd.to_numeric(total_summary_df["Direct Med Cost (M€)"], errors='coerce')
    total_summary_df["Direct Med Cost LCI (M€)"] = pd.to_numeric(total_summary_df["Direct Med Cost LCI (M€)"],
                                                                 errors='coerce')
    total_summary_df["Direct Med Cost UCI (M€)"] = pd.to_numeric(total_summary_df["Direct Med Cost UCI (M€)"],
                                                                 errors='coerce')

    # Filter out "TOTAL" and only include diseases
    plot_df_all = total_summary_df[total_summary_df["Disease"] != "TOTAL"].copy()

    # Iterate through unique pollutants to create separate plots
    pollutants = plot_df_all["Pollutant"].unique()

    for pollutant in pollutants:
        plot_df = plot_df_all[plot_df_all["Pollutant"] == pollutant].copy()

        # Sort by Year then Disease to ensure consistency on x-axis
        plot_df = plot_df.sort_values(["Year", "Disease"])

        years = sorted(plot_df["Year"].unique())
        diseases = sorted(plot_df["Disease"].unique())

        # Create categories for x-axis
        categories = []
        for yr in years:
            for dis in diseases:
                categories.append(f"{yr}\n{dis}")

        x = np.arange(len(categories))
        width = 0.35

        fig, ax = plt.subplots(figsize=(16, 8), dpi=140)

        for i, scenario in enumerate(["S2", "S3"]):
            medians, lcis, ucis = [], [], []

            for cat in categories:
                yr_val, dis_val = cat.split("\n")
                match = plot_df[
                    (plot_df["Scenario"] == scenario) &
                    (plot_df["Year"] == int(yr_val)) &
                    (plot_df["Disease"] == dis_val)
                    ]

                if not match.empty:
                    medians.append(match["Direct Med Cost (M€)"].values[0])
                    lcis.append(match["Direct Med Cost LCI (M€)"].values[0])
                    ucis.append(match["Direct Med Cost UCI (M€)"].values[0])
                else:
                    medians.append(0)
                    lcis.append(0)
                    ucis.append(0)

            positions = x + (i - 0.5) * width

            # Plot Median bars
            ax.bar(
                positions, medians, width,
                color=scenario_colors[scenario], edgecolor="black",
                linewidth=0.9, alpha=0.8, label=scenario
            )

            # Plot LCI/UCI as dotted lines
            for pos, lo, hi in zip(positions, lcis, ucis):
                ax.vlines(pos, lo, hi, color='black', linestyle='dotted', linewidth=1.5)

        # Add vertical line to separate 2030 and 2050
        sep_pos = len(diseases) - 0.5
        ax.axvline(x=sep_pos, color='grey', linestyle='--', linewidth=1.5, alpha=0.7)

        # X-axis labels
        ax.set_xticks(x)
        ax.set_xticklabels([cat.split("\n")[1] for cat in categories], fontsize=10, rotation=45, ha="right")

        # Secondary X-axis for Years
        for i, yr in enumerate(years):
            center_pos = (len(diseases) - 1) / 2 + i * len(diseases)
            ax.text(center_pos, -0.15, str(yr), transform=ax.get_xaxis_transform(),
                    ha='center', va='top', fontsize=14, weight='bold')

        ax.set_ylabel("Direct Medical Cost Avoided (M€)", fontsize=14, fontweight='bold')
        ax.set_title(f"Direct Medical Cost Avoided per Disease by Scenario and Year - {pollutant}", fontsize=18,
                     fontweight='bold')
        ax.grid(axis='y', linestyle="--", alpha=0.6)
        ax.legend(title="Scenario", fontsize=12, title_fontsize=14)

        # Save the plot
        save_path = os.path.join(path_fichier_expo, f"Medcost_disease_plot{pollutant}.png")
        plt.savefig(save_path, dpi=300, bbox_inches='tight')
        plt.tight_layout()
        plt.show()
else:
    print(f"File not found: {out_file}")


In [4]:
import os
import logging
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import Patch
from matplotlib.ticker import MaxNLocator

# ---------- Font / Logging ----------
rcParams['font.family'] = 'DejaVu Sans'
warnings.simplefilter(action='ignore', category=FutureWarning)
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

# ---------- Expected column order (for reference) ----------
cols_order = [
    "disease", "pollutant", "year", "age","avoided_cases_central",
    "avoided_cases_low", "avoided_cases_high", "mortality_avoided_central", "mortality_avoided_low", "mortality_avoided_high",
    "YLD_central", "YLD_low", "YLD_high",
    "YLL_central", "YLL_low", "YLL_high",
    "DALY_central", "DALY_low", "DALY_high"
]

# ---------- Age bins / disease age filters ----------
age_bins = [-np.inf, 9, 19, 29, 39, 49, 59, 69, 79, 89, np.inf]
age_labels = ['0-9', '10-19', '20-29', '30-39', '40-49', '50-59', '60-69', '70-79', '80-89', '90+']

disease_age_groups = {
    'Hypertension': (18, 99),
    'Lung Cancer': (35, 99),
    'Asthma in children': (0, 17),
    'Asthma in adult': (18, 99),
    'COPD': (40, 99),
    'ALRI in children': (0, 12),
    'Stroke': (35, 99),
    'IHD events': (30, 99),
    'Diabetes T2': (45, 99)
}

# ---------- Plot colors ----------
colors = ['#F28E2B', '#E15759', '#76B7B2', '#59A14F', '#EDC948', '#B07AA1', '#FF9DA7', '#9C755F', '#4E79A7']
mortality_color = '#2E4057'

# ---------- Which scenarios/pollutants/years ----------
scenarios = ["S2", "S3"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
years = ["2030", "2050"]

# ---------- Paths ----------
base_path = "data/2-output-data"
plot_dir = os.path.join(base_path, "plots")
os.makedirs(plot_dir, exist_ok=True)


# ---------- Helper: morbidity file path (expects morbidity_results.csv) ----------
def morbidity_csv_path(scenario, pollutant, year):
    for sc_variant in (scenario, scenario.lower(), scenario.upper()):
        p = os.path.join(base_path, sc_variant, pollutant, str(year), "morbidity_results.csv")
        if os.path.exists(p):
            return p
    return None


# ---------- Load all files into a single DataFrame ----------
frames = []
for pollutant in pollutants:
    for scenario in scenarios:
        for year in years:
            p = morbidity_csv_path(scenario, pollutant, year)
            if p is None:
                logging.warning(f"No morbidity_results.csv for {scenario}/{pollutant}/{year}")
                continue
            logging.info(f"Loading {p}")
            df = pd.read_csv(p, dtype=str)  # read as string to avoid surprises
            df.columns = [c.strip() for c in df.columns]
            # make sure common cols exist; create missing with NaN
            for c in cols_order:
                if c not in df.columns:
                    df[c] = np.nan
            # add metadata (ensure these exist/overwrite if necessary)
            df['scenario'] = scenario
            df['pollutant'] = pollutant
            df['year'] = str(year)
            frames.append(df)

if not frames:
    raise SystemExit("No morbidity files found. Check base_path and filenames.")
full = pd.concat(frames, ignore_index=True, sort=False)
logging.info(f"Combined dataframe shape: {full.shape}")


# ---------- Normalize numeric columns ----------
def first_existing(col_names, df):
    for c in col_names:
        if c in df.columns:
            return c
    return None


# Decide which columns to use for YLD/YLL/DALY and avoided_cases
yld_col = first_existing(['YLD_central', 'YLD', 'yld_central', 'yld'], full)
yll_col = first_existing(['YLL_central', 'YLL', 'yll_central', 'yll'], full)
daly_col = first_existing(['DALY_central', 'DALY', 'daly_central', 'daly'], full)
avoided_cases_col = first_existing(['avoided_cases_central', 'avoided_cases'], full)
age_col = first_existing(['age', 'Age', 'AGE'], full)
disease_col = first_existing(['disease', 'Disease', 'cause'], full)

# Ensure we have required columns
if age_col is None:
    raise SystemExit("No age column found in morbidity files.")
if disease_col is None:
    raise SystemExit("No disease column found in morbidity files.")

# convert types
full[age_col] = pd.to_numeric(full[age_col], errors='coerce')
for c in [yld_col, yll_col, daly_col, avoided_cases_col]:
    if c is not None:
        full[c] = pd.to_numeric(full[c], errors='coerce').fillna(0)

# Also ensure canonical names exist populated from chosen columns
for name, src in [('YLD', yld_col), ('YLL', yll_col), ('DALY', daly_col), ('avoided_cases', avoided_cases_col)]:
    if name not in full.columns or full[name].isna().all():
        full[name] = full[src].fillna(0) if src is not None else 0

# canonicalize column names for downstream use
full = full.rename(columns={age_col: 'age', disease_col: 'disease'})

# strip disease strings
full['disease'] = full['disease'].astype(str).str.strip()

# Filter disease entries using disease_age_groups
filtered_list = []
for disease_key, (min_age, max_age) in disease_age_groups.items():
    mask = full['disease'].str.contains(disease_key, case=False, na=False) | (full['disease'] == disease_key)
    df_d = full[mask].copy()
    if df_d.empty:
        logging.info(f"No rows found for disease: {disease_key}")
        continue
    df_d = df_d[(df_d['age'] >= min_age) & (df_d['age'] <= max_age)].copy()
    if df_d.empty:
        logging.info(f"No age-eligible rows for {disease_key} after filtering")
        continue
    df_d['disease'] = disease_key  # canonical name
    filtered_list.append(df_d)

if not filtered_list:
    raise SystemExit("No disease data after filtering. Check disease names in the morbidity files.")

filtered = pd.concat(filtered_list, ignore_index=True, sort=False)

# Age groups
filtered['age_group'] = pd.cut(filtered['age'], bins=age_bins, labels=age_labels, right=True)

#Prefer central variants if present
def choose_col(df, primary, fallback_list):
    if primary in df.columns:
        return primary
    for fb in fallback_list:
        if fb in df.columns:
            return fb
    return None


yld_use = choose_col(filtered, "YLD_central", ["YLD"])
yll_use = choose_col(filtered, "YLL_central", ["YLL"])
daly_use = choose_col(filtered, "DALY_central", ["DALY"])

for col in [yld_use, yll_use, daly_use]:
    if col and col in filtered.columns:
        filtered[col] = pd.to_numeric(filtered[col], errors='coerce').fillna(0)
    else:
        new_col = col if col else "TMP"
        filtered[new_col] = 0

# Aggregate with chosen columns
agg = filtered.groupby(['pollutant', 'year', 'scenario', 'age_group', 'disease'], dropna=False).agg({
    yld_use: 'sum',
    yll_use: 'sum',
    daly_use: 'sum'
}).reset_index().rename(columns={
    yld_use: 'YLD_plot',
    yll_use: 'YLL_plot',
    daly_use: 'DALY_plot'
})

# Plotting: one pollutant per figure, 2x2 (years x scenarios)
for pollutant in agg['pollutant'].unique():
    dis_impact = (
        agg[agg['pollutant'] == pollutant]
        .groupby('disease', as_index=False)['YLD_plot']
        .sum()
    )
    present_diseases = dis_impact.loc[dis_impact['YLD_plot'] > 0, 'disease'].tolist()
    if not present_diseases:
        present_diseases = dis_impact['disease'].unique().tolist()

    disease_color_map = {
        d: colors[i % len(colors)]
        for i, d in enumerate(sorted(present_diseases))
    }

    fig, axes = plt.subplots(
        nrows=2, ncols=2,
        figsize=(14, 10),
        sharey=False
    )

    fig.suptitle(
        f"Contribution of mortality and morbidity to DALYs — {pollutant}",
        fontsize=16,
        weight="bold",
        y=0.98
    )

    for i, year in enumerate(years):
        for j, scenario in enumerate(scenarios):
            ax = axes[i, j]

            sub = agg[
                (agg['pollutant'] == pollutant) &
                (agg['year'] == year) &
                (agg['scenario'] == scenario)
                ]

            if sub.empty:
                ax.text(0.5, 0.5,f"No data\n{scenario} {year}",ha='center', va='center', fontsize=11)
                ax.set_axis_off()
                continue

            yll_by_age = (sub.groupby('age_group')['YLL_plot'].sum()
                .reindex(age_labels, fill_value=0)
            )

            yld_pivot = (
                sub.pivot_table(
                    index='age_group',
                    columns='disease',
                    values='YLD_plot',
                    aggfunc='sum'
                )
                .reindex(index=age_labels)
                .fillna(0)
            )

            x = np.arange(len(age_labels))
            width = 0.8

            ax.bar(
                x,
                yll_by_age.values,
                width=width,
                color=mortality_color,
                label="Mortality (YLL)"
            )
            bottom = yll_by_age.values.copy()
            for disease in yld_pivot.columns:
                vals = yld_pivot[disease].values
                if np.allclose(vals, 0):
                    continue
                ax.bar(
                    x,
                    vals,
                    bottom=bottom,
                    width=width,
                    color=disease_color_map.get(disease, "#999999"),
                    label=disease
                )
                bottom += vals

            # ---- Axis formatting
            ax.set_title(f"{scenario} — {year}", fontsize=13, weight="bold")
            ax.set_ylabel("DALYs averted", fontsize=12)
            ax.grid(axis="y", linestyle="--", alpha=0.4)
            ax.set_axisbelow(True)
            # Adjust the number of ticks on the y-axis
            ax.yaxis.set_major_locator(MaxNLocator(nbins=10))
            ax.set_xticks(x)
            if i == 1:
                ax.set_xticklabels(age_labels, rotation=40, ha="right", fontsize=10)
                ax.set_xlabel("Age group", fontsize=11)
            else:
                ax.set_xticklabels([])

    # ---------- Legend (boxed, bottom, centered) ----------
    legend_handles = [Patch(facecolor=mortality_color, label="Mortality (YLL)")]
    legend_handles += [
        Patch(facecolor=disease_color_map[d], label=d)
        for d in sorted(disease_color_map)
    ]

    fig.legend(
        handles=legend_handles,
        labels=[h.get_label() for h in legend_handles],
        loc="lower center",
        bbox_to_anchor=(0.5, -0.02),
        ncol=4,
        frameon=True,
        framealpha=0.95,
        edgecolor="black",
        fontsize=11,
        title="Diseases",
        title_fontsize=12
    )

    plt.tight_layout(rect=[0, 0.12, 1, 0.95])

    out_file = os.path.join(plot_dir, f"DALY_by_agegroup_{pollutant}.png")
    plt.savefig(out_file, dpi=300, bbox_inches="tight")
    logging.info(f"Saved figure: {out_file}")
    plt.close()

logging.info("Done — DALY plots created.")


2026-01-23 14:17:59,146 - INFO - Loading data/2-output-data\S2\ug_PM25_RH50\2030\morbidity_results.csv
2026-01-23 14:17:59,159 - INFO - Loading data/2-output-data\S2\ug_PM25_RH50\2050\morbidity_results.csv
2026-01-23 14:17:59,167 - INFO - Loading data/2-output-data\S3\ug_PM25_RH50\2030\morbidity_results.csv
2026-01-23 14:17:59,167 - INFO - Loading data/2-output-data\S3\ug_PM25_RH50\2050\morbidity_results.csv
2026-01-23 14:17:59,167 - INFO - Loading data/2-output-data\S2\ug_NO2\2030\morbidity_results.csv
2026-01-23 14:17:59,183 - INFO - Loading data/2-output-data\S2\ug_NO2\2050\morbidity_results.csv
2026-01-23 14:17:59,183 - INFO - Loading data/2-output-data\S3\ug_NO2\2030\morbidity_results.csv
2026-01-23 14:17:59,183 - INFO - Loading data/2-output-data\S3\ug_NO2\2050\morbidity_results.csv
2026-01-23 14:17:59,183 - INFO - Combined dataframe shape: (2120, 21)
2026-01-23 14:18:00,562 - INFO - Saved figure: data/2-output-data\plots\DALY_by_agegroup_ug_NO2.png
2026-01-23 14:18:02,031 - INFO

In [11]:
import os
import logging
import warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import rcParams
from matplotlib.patches import Patch

#  Font / Logging
rcParams["font.family"] = "DejaVu Sans"
warnings.simplefilter(action="ignore", category=FutureWarning)
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

# Expected column order (for reference)
cols_order = [
    "disease",
    "pollutant",
    "year",
    "age",
    "avoided_cases_central",
    "avoided_cases_low",
    "avoided_cases_high",
    "mortality_avoided_central",
    "mortality_avoided_low",
    "mortality_avoided_high",
    "YLD_central",
    "YLD_low",
    "YLD_high",
    "YLL_central",
    "YLL_low",
    "YLL_high",
    "DALY_central",
    "DALY_low",
    "DALY_high",
]

# Age bins / disease age filters
age_bins = [-np.inf, 9, 19, 29, 39, 49, 59, 69, 79, 89, np.inf]
age_labels = ["0-9", "10-19", "20-29", "30-39", "40-49", "50-59", "60-69", "70-79", "80-89", "90+"]

disease_age_groups = {
    "Hypertension": (18, 99),
    "Lung Cancer": (35, 99),
    "Asthma in children": (0, 17),
    "Asthma in adult": (18, 99),
    "COPD": (40, 99),
    "ALRI in children": (0, 12),
    "Stroke": (35, 99),
    "IHD events": (30, 99),
    "Diabetes T2": (45, 99),
}

# Plot colors
colors = ["#F28E2B", "#E15759", "#76B7B2", "#59A14F", "#EDC948", "#B07AA1", "#FF9DA7", "#D4C23D", "#4E79A7"]
mortality_color = "#2E4057"

scenarios = ["S2", "S3"]
pollutants = ["ug_PM25_RH50", "ug_NO2"]
years = ["2030", "2050"]

# Paths
base_path = "data/2-output-data"
plot_dir = os.path.join(base_path, "plots")
os.makedirs(plot_dir, exist_ok=True)


# Helper: morbidity file path (expects morbidity_results.csv) ----------
def morbidity_csv_path(scenario, pollutant, year):
    for sc_variant in (scenario, scenario.lower(), scenario.upper()):
        p = os.path.join(base_path, sc_variant, pollutant, str(year), "morbidity_results.csv")
        if os.path.exists(p):
            return p
    return None


# Load all files into a single DataFrame ----------
frames = []
for pollutant in pollutants:
    for scenario in scenarios:
        for year in years:
            p = morbidity_csv_path(scenario, pollutant, year)
            if p is None:
                logging.warning(f"No morbidity_results.csv for {scenario}/{pollutant}/{year}")
                continue
            logging.info(f"Loading {p}")
            df = pd.read_csv(p, dtype=str)  # read as string to avoid surprises
            df.columns = [c.strip() for c in df.columns]
            # make sure common cols exist; create missing with NaN
            for c in cols_order:
                if c not in df.columns:
                    df[c] = np.nan
            # add metadata (ensure these exist/overwrite if necessary)
            df["scenario"] = scenario
            df["pollutant"] = pollutant
            df["year"] = str(year)
            frames.append(df)

if not frames:
    raise SystemExit("No morbidity files found. Check base_path and filenames.")
full = pd.concat(frames, ignore_index=True, sort=False)
logging.info(f"Combined dataframe shape: {full.shape}")


# ---------- Normalize numeric columns ----------
def first_existing(col_names, df):
    for c in col_names:
        if c in df.columns:
            return c
    return None


# Decide which columns to use for YLD/YLL/DALY and avoided_cases
yld_col = first_existing(["YLD_central", "YLD", "yld_central", "yld"], full)
yll_col = first_existing(["YLL_central", "YLL", "yll_central", "yll"], full)
daly_col = first_existing(["DALY_central", "DALY", "daly_central", "daly"], full)
avoided_cases_col = first_existing(["avoided_cases_central", "avoided_cases"], full)
age_col = first_existing(["age", "Age", "AGE"], full)
disease_col = first_existing(["disease", "Disease", "cause"], full)

# Ensure we have required columns
if age_col is None:
    raise SystemExit("No age column found in morbidity files.")
if disease_col is None:
    raise SystemExit("No disease column found in morbidity files.")

# convert types
full[age_col] = pd.to_numeric(full[age_col], errors="coerce")
for c in [yld_col, yll_col, daly_col, avoided_cases_col]:
    if c is not None:
        full[c] = pd.to_numeric(full[c], errors="coerce").fillna(0)

# Also ensure the names exist populated from chosen columns
for name, src in [("YLD", yld_col), ("YLL", yll_col), ("DALY", daly_col), ("avoided_cases", avoided_cases_col)]:
    if name not in full.columns or full[name].isna().all():
        full[name] = full[src].fillna(0) if src is not None else 0

# rename col
full = full.rename(columns={age_col: "age", disease_col: "disease"})

# strip disease strings
full["disease"] = full["disease"].astype(str).str.strip()

# Filter disease entries using disease_age_groups
filtered_list = []
for disease_key, (min_age, max_age) in disease_age_groups.items():
    mask = full["disease"].str.contains(disease_key, case=False, na=False) | (full["disease"] == disease_key)
    df_d = full[mask].copy()
    if df_d.empty:
        logging.info(f"No rows found for disease: {disease_key}")
        continue
    df_d = df_d[(df_d["age"] >= min_age) & (df_d["age"] <= max_age)].copy()
    if df_d.empty:
        logging.info(f"No age-eligible rows for {disease_key} after filtering")
        continue
    df_d["disease"] = disease_key  # canonical name
    filtered_list.append(df_d)

if not filtered_list:
    raise SystemExit("No disease data after filtering. Check disease names in the morbidity files.")

filtered = pd.concat(filtered_list, ignore_index=True, sort=False)

# Age groups
filtered["age_group"] = pd.cut(filtered["age"], bins=age_bins, labels=age_labels, right=True)


# Prefer central variants if present
def choose_col(df, primary, fallback_list):
    if primary in df.columns:
        return primary
    for fb in fallback_list:
        if fb in df.columns:
            return fb
    return None


yld_use = choose_col(filtered, "YLD_central", ["YLD"])
yll_use = choose_col(filtered, "YLL_central", ["YLL"])
daly_use = choose_col(filtered, "DALY_central", ["DALY"])

for col in [yld_use, yll_use, daly_use]:
    if col and col in filtered.columns:
        filtered[col] = pd.to_numeric(filtered[col], errors="coerce").fillna(0)
    else:
        new_col = col if col else "TMP"
        filtered[new_col] = 0

# Aggregate with chosen columns
agg = (
    filtered.groupby(["pollutant", "year", "scenario", "age_group", "disease"], dropna=False)
    .agg({yld_use: "sum", yll_use: "sum", daly_use: "sum"})
    .reset_index()
    .rename(columns={yld_use: "YLD_plot", yll_use: "YLL_plot", daly_use: "DALY_plot"})
)

#  Plotting with broken Y-axis: YLL (top) + YLD (bottom)
for pollutant in agg["pollutant"].unique():

    # diseases actually contributing YLD
    dis_impact = agg[agg["pollutant"] == pollutant].groupby("disease", as_index=False)["YLD_plot"].sum()
    present_diseases = dis_impact.loc[dis_impact["YLD_plot"] > 0, "disease"].tolist()
    if not present_diseases:
        present_diseases = dis_impact["disease"].unique().tolist()

    disease_color_map = {d: colors[i % len(colors)] for i, d in enumerate(sorted(present_diseases))}

    fig = plt.figure(figsize=(14, 10))
    fig.suptitle(
        f"",
        fontsize=16,
        weight="bold",
        y=0.98,
    )

    # Grid: 2 rows (years) × 2 cols (scenarios),
    # each cell split vertically into YLL (top) and YLD (bottom)
    outer = fig.add_gridspec(2, 2, wspace=0.15, hspace=0.25)

    for i, year in enumerate(years):
        for j, scenario in enumerate(scenarios):

            sub = agg[(agg["pollutant"] == pollutant) & (agg["year"] == year) & (agg["scenario"] == scenario)]

            if sub.empty:
                ax = fig.add_subplot(outer[i, j])
                ax.text(0.5, 0.5, f"No data\n{scenario} {year}", ha="center", va="center")
                ax.set_axis_off()
                continue

            # Data prep
            yll_by_age = sub.groupby("age_group")["YLL_plot"].sum().reindex(age_labels, fill_value=0)

            yld_pivot = (
                sub.pivot_table(index="age_group", columns="disease", values="YLD_plot", aggfunc="sum")
                .reindex(index=age_labels)
                .fillna(0)
            )

            x = np.arange(len(age_labels))
            width = 0.8

            # Create broken axes
            gs_inner = outer[i, j].subgridspec(2, 1, height_ratios=[3, 1], hspace=0.05)

            ax_top = fig.add_subplot(gs_inner[0])
            ax_bot = fig.add_subplot(gs_inner[1], sharex=ax_top)

            # TOP: YLL ONLY
            ax_top.bar(x, yll_by_age.values, width=width, color=mortality_color)

            ax_top.set_ylabel("DALYs averted", fontsize=11)
            ax_top.set_title(f"{scenario} — {year}", fontsize=13, weight="bold")
            ax_top.grid(axis="y", linestyle="--", alpha=0.4)
            ax_top.tick_params(labelbottom=False)

            # dynamic YLL limits
            yll_max = yll_by_age.max()
            ax_top.set_ylim(0, yll_max * 1.15 if yll_max > 0 else 1)

            # BOTTOM: YLD STACKED
            bottom = np.zeros(len(age_labels))
            for disease in yld_pivot.columns:
                vals = yld_pivot[disease].values
                if np.allclose(vals, 0):
                    continue
                ax_bot.bar(
                    x,
                    vals,
                    bottom=bottom,
                    width=width,
                    color=disease_color_map.get(disease, "#999999"),
                )
                bottom += vals

            ax_bot.grid(axis="y", linestyle="--", alpha=0.4)

            # dynamic YLD limits (zoom)
            yld_max = bottom.max()
            ax_bot.set_ylim(0, yld_max * 1.3 if yld_max > 0 else 1)

            # Set more ticks on the y-axis for the lower plot
            from matplotlib.ticker import MaxNLocator

            ax_bot.yaxis.set_major_locator(MaxNLocator(nbins=6))

            ax_bot.set_xticks(x)
            ax_bot.set_xticklabels(age_labels, rotation=40, ha="right", fontsize=10)
            if i == 1:
                ax_bot.set_xlabel("", fontsize=11)

            # Break marks
            d = 0.015
            kwargs = dict(transform=ax_top.transAxes, color="k", clip_on=False)
            ax_top.plot((-d, +d), (-d, +d), **kwargs)
            ax_top.plot((1 - d, 1 + d), (-d, +d), **kwargs)

            kwargs.update(transform=ax_bot.transAxes)
            ax_bot.plot((-d, +d), (1 - d, 1 + d), **kwargs)
            ax_bot.plot((1 - d, 1 + d), (1 - d, 1 + d), **kwargs)

    # ---------- Legend ----------
    legend_handles = [Patch(facecolor=mortality_color, label="Mortality")]
    legend_handles += [Patch(facecolor=disease_color_map[d], label=d) for d in sorted(disease_color_map)]

    fig.legend(
        handles=legend_handles,
        loc="lower center",
        bbox_to_anchor=(0.5, 0.01),
        ncol=8,
        frameon=True,
        framealpha=0.95,
        edgecolor="black",
        fontsize=11,
        title="Diseases",
        title_fontsize=11,
    )

    # Avoid tight_layout warning with complex nested GridSpecs
    fig.subplots_adjust(left=0.06, right=0.98, top=0.93, bottom=0.12, wspace=0.16, hspace=0.22)

    out_file = os.path.join(plot_dir, f"DALY_by_agegroup_broken_{pollutant}.png")
    fig.savefig(out_file, dpi=300, bbox_inches="tight")
    logging.info(f"Saved figure: {out_file}")
    plt.close(fig)


2026-01-23 14:35:47,510 - INFO - Loading data/2-output-data\S2\ug_PM25_RH50\2030\morbidity_results.csv
2026-01-23 14:35:47,521 - INFO - Loading data/2-output-data\S2\ug_PM25_RH50\2050\morbidity_results.csv
2026-01-23 14:35:47,521 - INFO - Loading data/2-output-data\S3\ug_PM25_RH50\2030\morbidity_results.csv
2026-01-23 14:35:47,521 - INFO - Loading data/2-output-data\S3\ug_PM25_RH50\2050\morbidity_results.csv
2026-01-23 14:35:47,521 - INFO - Loading data/2-output-data\S2\ug_NO2\2030\morbidity_results.csv
2026-01-23 14:35:47,537 - INFO - Loading data/2-output-data\S2\ug_NO2\2050\morbidity_results.csv
2026-01-23 14:35:47,537 - INFO - Loading data/2-output-data\S3\ug_NO2\2030\morbidity_results.csv
2026-01-23 14:35:47,545 - INFO - Loading data/2-output-data\S3\ug_NO2\2050\morbidity_results.csv
2026-01-23 14:35:47,546 - INFO - Combined dataframe shape: (2120, 21)
2026-01-23 14:35:49,396 - INFO - Saved figure: data/2-output-data\plots\DALY_by_agegroup_broken_ug_NO2.png
2026-01-23 14:35:50,806